# CLIP-Guided Test-Time Optimization with GigaTok

This notebook demonstrates CLIP-guided image editing and token interpretability using GigaTok's 1D VQ tokenizer with hierarchical ViT encoder/decoder.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

In [ ]:
import sys
if IN_COLAB:
    !git clone -q https://github.com/sbeeredd04/sandbox.git
    !pip install -q --progress-bar off jaxtyping open_clip_torch omegaconf timm
    sys.path.insert(0, "sandbox/token-opt")
    sys.path.insert(0, "sandbox/GigaTok")
else:
    # For local development
    sys.path.insert(0, "/home/sbeeredd/sandbox/token-opt")
    sys.path.insert(0, "/home/sbeeredd/sandbox/GigaTok")

print("Python path:")
for p in sys.path[:5]:
    print(f"  - {p}")

In [ ]:
import os
# Set this environment for deterministic execution
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [ ]:
import torch
# Enable for deterministic algorithms
torch.use_deterministic_algorithms(True, warn_only=False)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from pathlib import Path
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as v2
import torchvision.transforms.v2.functional as tvf
from torchvision.datasets import ImageNet
from einops import rearrange

In [ ]:
from tto.test_time_opt import (
    TestTimeOpt,
    TestTimeOptConfig,
    CLIPObjective,
)

In [ ]:
gpus = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = gpus
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Using device: {device}, num_gpus: {num_gpus}")

## Utils

In [ ]:
def load_img(path, device=None):
    if IN_COLAB:
        path = Path("sandbox/token-opt/notebooks") / Path(path)
    img = (1. / 255.) * torch.from_numpy(
        np.array(Image.open(path)).astype(np.float32)
    ).permute(2, 0, 1)
    img = tvf.resize(img, 256)
    img = tvf.center_crop(img, 256)
    img = img.unsqueeze(0)
    if device is not None:
        img = img.to(device)
    return img

def display_image(*tensors):
    tensors = [255. * t.squeeze() for t in tensors]
    img = Image.fromarray(rearrange(
        tensors, "b c h w -> h (b w) c"
    ).to("cpu", dtype=torch.uint8).numpy())
    display(img)

def opt_callback(info):
    if info.i % 50 == 0:
        print(f"i = {info.i}")
        print("  CLIP score =", "\t".join(
            map(lambda l: f"{-l:.3f}", info.loss))
        )
        imgs = tto.decode(info.tokens).clamp(0., 1.)
        display_image(*imgs)

# Set up the objective function

In [ ]:
# Use CLIP similarity maximization objective
objective = CLIPObjective(num_augmentations=8, cfg_scale=1.2)

# Set prompt
objective.prompt = [
    "a photo of a tiger",
    "a photo of a husky",
    "a photo of a sparrow",
]

# Optionally set a negative prompt
# Note: also need to set cfg_scale > 1 in CLIPObjective if using this!
objective.neg_prompt = "bad, low-res, unnatural"

# Configure test time optimization with GigaTok

GigaTok is a 1D VQ tokenizer with hierarchical ViT encoder/decoder architecture:
- **BL256**: Base encoder, Large decoder, 256 tokens (smaller, faster)
- **XLXXL256**: XL encoder, XXL decoder, 256 tokens (3B params, best quality)

The tokenizer compresses 256×256 images into 256 discrete tokens using:
1. CNN encoder → spatial features
2. ViT 2D→1D encoder → 256 latent tokens
3. Vector quantization → discrete codes (16,384 codebook entries)
4. ViT 1D→2D decoder → spatial features
5. CNN decoder → reconstructed image

In [ ]:
tto_config = TestTimeOptConfig(
    # Use GigaTok tokenizer
    # Format: "gigatok:MODEL_CONFIG" or "gigatok:MODEL_CONFIG:CHECKPOINT_PATH"
    # Available configs: BL256, XLXXL256
    titok_checkpoint="gigatok:BL256",  # Use BL256 for faster inference
    
    # Optimize in continuous space (before quantization)
    optimize_post_quantization_tokens=True,
    
    # Optimization parameters
    num_iter=1000,
    ema_decay=0.98,
    lr=0.1,
    enable_amp=True,
    reg_weight=0.025,
    reg_type="seed",
)
tto = TestTimeOpt(tto_config, objective).to(device)

# Load seed images

In [ ]:
# Load seed image
img = torch.cat([
    load_img("ILSVRC2012_val_00008636.png", device),
    load_img("ILSVRC2012_val_00008636.png", device),
    load_img("ILSVRC2012_val_00010240.png", device),
], dim=0)

# Alternatively, initialize directly from given tokens (e.g. randomly
# sampled), but this is disabled when setting `seed_tokens = None`.
seed_tokens = None

# Run Optimization

In [ ]:
print("Seed")
display_image(*img)

# Run optimization
torch.manual_seed(0)
img_opt = tto(
    seed=img if seed_tokens is None else None,
    seed_tokens=seed_tokens,
    callback=opt_callback
)

# Token Interpretability: Token Swapping

Let's explore what each token represents by swapping tokens between two images.

GigaTok uses **256 1D tokens** to represent an image. Unlike 2D tokenizers, these tokens are computed via cross-attention and don't have direct spatial correspondence, but we can still analyze their influence.

In [ ]:
from tto.gigatok_wrapper import load_gigatok_model
import imageio
from pathlib import Path
from IPython.display import Image as IPImage

# Load GigaTok model
gigatok_model = load_gigatok_model('BL256')
gigatok_model.eval()
gigatok_model.to(device)

In [ ]:
def encode_tokens_gigatok(model, img, return_quantized=False):
    with torch.no_grad():
        # Encode: (b, 3, 256, 256) -> (b, codebook_embed_dim, 1, num_latent_tokens)
        z = model.encode(img)
        
        if return_quantized:
            # Quantize tokens
            z, _ = model.quantize(z)
        
        return z


def decode_tokens_gigatok(model, tokens, are_quantized=False):
    with torch.no_grad():
        # If not already quantized, quantize before decoding
        if not are_quantized:
            tokens, _ = model.quantize(tokens)
        
        # Decode: (b, codebook_embed_dim, 1, num_latent_tokens) -> (b, 3, 256, 256)
        img = model.decode(tokens)
        return img


def swap_token_range(base_tokens, source_tokens, start_idx, end_idx):
    swapped = base_tokens.clone()
    swapped[:, :, :, start_idx:end_idx] = source_tokens[:, :, :, start_idx:end_idx]
    return swapped

In [ ]:
def save_img_tensor(tensor, path):
    tensor = tensor.squeeze().clamp(0., 1.) * 255.
    img_array = tensor.permute(1, 2, 0).cpu().to(dtype=torch.uint8).numpy()
    img = Image.fromarray(img_array)
    img.save(path)
    return img_array


def concat_images(img1, img2):
    img1_np = img1.squeeze().clamp(0., 1.) * 255.
    img1_np = img1_np.permute(1, 2, 0).cpu().to(dtype=torch.uint8).numpy()

    img2_np = img2.squeeze().clamp(0., 1.) * 255.
    img2_np = img2_np.permute(1, 2, 0).cpu().to(dtype=torch.uint8).numpy()

    concat_images = np.concatenate([img1_np, img2_np], axis=1)
    return concat_images


# Create output directory
output_dir = Path("gigatok_token_swap_frames")
output_dir.mkdir(exist_ok=True)

# Set whether to quantize tokens immediately
use_quantized_tokens = True

# Load two images for token swapping
img1 = load_img("ILSVRC2012_val_00008636.png", device)
img2 = load_img("ILSVRC2012_val_00010240.png", device)

# Encode to tokens
tokens1_original = encode_tokens_gigatok(gigatok_model, img1, return_quantized=use_quantized_tokens)
tokens2_original = encode_tokens_gigatok(gigatok_model, img2, return_quantized=use_quantized_tokens)

# Get token dimensions
b, d, _, n = tokens1_original.shape
print(f"Token shape: {tokens1_original.shape}")
print(f"Number of tokens: {n}")
print(f"Codebook dimension: {d}")
print(f"{'='*60}\n")

# Collect frames for GIF
frames = []

# Swap tokens progressively (in chunks for efficiency)
chunk_size = 8  # Swap 8 tokens at a time for smoother animation
num_chunks = (n + chunk_size - 1) // chunk_size

print(f"Creating progressive token swap animation ({num_chunks} frames)...")

for chunk_idx in range(num_chunks + 1):
    end_idx = min(chunk_idx * chunk_size, n)
    print(f"Tokens swapped: {end_idx}/{n}...", end="\r")
    
    # Swap tokens [0:end_idx] between the two images
    pair1_swapped = swap_token_range(tokens1_original, tokens2_original, 0, end_idx)
    pair2_swapped = swap_token_range(tokens2_original, tokens1_original, 0, end_idx)
    
    # Decode both pairs
    img_pair1 = decode_tokens_gigatok(gigatok_model, pair1_swapped, are_quantized=use_quantized_tokens)
    img_pair2 = decode_tokens_gigatok(gigatok_model, pair2_swapped, are_quantized=use_quantized_tokens)
    
    # Create side-by-side images
    pair1_concat = concat_images(img1, img_pair1)
    pair2_concat = concat_images(img2, img_pair2)
    
    # Stack both pairs vertically
    combined_frame = np.concatenate([pair1_concat, pair2_concat], axis=0)
    
    frames.append(combined_frame)

print(f"\n{'='*60}")
print(f"Progressive token swapping complete!")
print(f"{'='*60}\n")

# Save as GIF
gif_path_combined = output_dir / "token_swap_progressive.gif"
imageio.mimsave(gif_path_combined, frames, duration=100, loop=0)

print(f"GIF saved at {gif_path_combined}")

# Display the combined GIF
print("\nProgressive token swap animation:")
display(IPImage(filename=str(gif_path_combined)))

## Individual Token Importance

Now let's see the effect of swapping individual tokens (or small groups) to understand token-level semantics.

In [ ]:
# Create output directory
output_dir = Path("gigatok_individual_token_swap")
output_dir.mkdir(exist_ok=True)

# Token groups to visualize (beginning, middle, end)
token_positions = [
    (0, 8, "First 8 tokens"),
    (64, 72, "Tokens 64-72"),
    (128, 136, "Middle tokens (128-136)"),
    (192, 200, "Tokens 192-200"),
    (248, 256, "Last 8 tokens"),
]

print("Testing individual token group swaps...\n")

frames = []

for start_idx, end_idx, description in token_positions:
    print(f"Processing: {description}")
    
    # Swap token range
    pair1_swapped = swap_token_range(tokens1_original, tokens2_original, start_idx, end_idx)
    pair2_swapped = swap_token_range(tokens2_original, tokens1_original, start_idx, end_idx)
    
    # Decode both pairs
    img_pair1 = decode_tokens_gigatok(gigatok_model, pair1_swapped, are_quantized=use_quantized_tokens)
    img_pair2 = decode_tokens_gigatok(gigatok_model, pair2_swapped, are_quantized=use_quantized_tokens)
    
    # Create side-by-side images
    pair1_concat = concat_images(img1, img_pair1)
    pair2_concat = concat_images(img2, img_pair2)
    
    # Stack both pairs vertically
    combined_frame = np.concatenate([pair1_concat, pair2_concat], axis=0)
    
    # Save individual frame
    frame_pil = Image.fromarray(combined_frame)
    frames.append(np.array(frame_pil))
    
    frame_path = output_dir / f"swap_tokens_{start_idx}_{end_idx}.png"
    frame_pil.save(frame_path)
    print(f"  Saved: {frame_path}")

print(f"\n{'='*60}")
print(f"Individual token swap complete!")
print(f"{'='*60}\n")

# Save as GIF
gif_path_combined = output_dir / "individual_token_swaps.gif"
imageio.mimsave(gif_path_combined, frames, duration=1500, loop=0)  # 1.5 sec per frame

print(f"GIF saved at {gif_path_combined}")

# Display the GIF
print("\nIndividual token group swap animation:")
display(IPImage(filename=str(gif_path_combined)))

In [1]:
import shutil
from pathlib import Path
from tqdm import tqdm

# Import ImageNet class names
import sys
sys.path.insert(0, "/home/sbeeredd/sandbox/ImageFolder")
from imagenet_classes import imagenet_idx2classname

# Path to ImageNet validation dataset
imagenet_val_path = Path("/home/hongjunchoi/imagenet/val")

# Output directory for samples
output_dir = Path("imagenet_samples")
output_dir.mkdir(exist_ok=True)

# Number of samples per class
samples_per_class = 5

print(f"Collecting {samples_per_class} samples from each ImageNet class...")
print(f"Source: {imagenet_val_path}")
print(f"Destination: {output_dir.absolute()}")
print("=" * 60)

# Get all class directories and create synset to index mapping
class_dirs = sorted([d for d in imagenet_val_path.iterdir() if d.is_dir()])
synset_to_idx = {d.name: i for i, d in enumerate(class_dirs)}
total_classes = len(class_dirs)

print(f"Found {total_classes} classes")
print(f"Total images to collect: {total_classes * samples_per_class}")
print("=" * 60)

# Collect samples from each class
collected_count = 0
skipped_count = 0

for class_dir in tqdm(class_dirs, desc="Processing classes"):
    synset = class_dir.name
    
    # Get class index and name
    class_idx = synset_to_idx[synset]
    class_name = imagenet_idx2classname[class_idx]
    
    # Clean class name (take first name, remove special chars)
    clean_class_name = class_name.split(',')[0].replace(' ', '_').replace("'", "")
    
    # Create folder name: <class_name>_<synset>
    folder_name = f"{clean_class_name}_{synset}"
    
    # Get all images in this class
    image_files = sorted(list(class_dir.glob("*.JPEG")))
    
    # Skip if not enough images
    if len(image_files) < samples_per_class:
        print(f"\nWarning: Class {folder_name} has only {len(image_files)} images (skipping)")
        skipped_count += 1
        continue
    
    # Take first N samples
    sample_images = image_files[:samples_per_class]
    
    # Create class directory in output with new naming format
    class_output_dir = output_dir / folder_name
    class_output_dir.mkdir(exist_ok=True)
    
    # Copy sample images
    for img_path in sample_images:
        dest_path = class_output_dir / img_path.name
        if not dest_path.exists():
            shutil.copy2(img_path, dest_path)
            collected_count += 1

print("\n" + "=" * 60)
print(f"Collection complete!")
print(f"Total classes processed: {total_classes - skipped_count}/{total_classes}")
print(f"Total images collected: {collected_count}")
print(f"Output directory: {output_dir.absolute()}")
print("=" * 60)

# Show a few example folder names
example_folders = sorted([d.name for d in output_dir.iterdir() if d.is_dir()])[:5]
print(f"\nExample folder names:")
for folder in example_folders:
    print(f"  - {folder}")

Source: /home/hongjunchoi/imagenet/val
Destination: /home/sbeeredd/sandbox/token-opt/notebooks/imagenet_samples
Found 1000 classes
Total images to collect: 5000


Processing classes:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing classes: 100%|██████████| 1000/1000 [00:01<00:00, 725.79it/s]


Collection complete!
Total classes processed: 1000/1000
Total images collected: 5000
Output directory: /home/sbeeredd/sandbox/token-opt/notebooks/imagenet_samples

Example folder names:
  - Afghan_hound_n02088094
  - African_chameleon_n01694178
  - African_crocodile_n01697457
  - African_elephant_n02504458
  - African_grey_n01817953
